In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


pasta_alertario = Path(
    "/home/noemi/atmoseer/data/alertario/pluviometricos_parquet"
)

arquivo_estacoes = Path(
    "/home/noemi/atmoseer/data/alertario/estacoes_alertario.csv"
)

arquivo_grade_radar = Path(
    "sumare_radar_latlon_grid.npz"
)

arquivo_saida = Path(
    "mapeamento_pixel_estacao_alertario.csv"
)

print("Pasta pluviométricos:", pasta_alertario)
print("Cadastro de estações:", arquivo_estacoes)
print("Grade do radar:", arquivo_grade_radar)

Pasta pluviométricos: /home/noemi/atmoseer/data/alertario/pluviometricos_parquet
Cadastro de estações: /home/noemi/atmoseer/data/alertario/estacoes_alertario.csv
Grade do radar: sumare_radar_latlon_grid.npz


In [3]:
stations_df = pd.read_csv(
    arquivo_estacoes
)

print("Colunas:")
print(stations_df.columns.tolist())

print("\nQuantidade de estações cadastradas:")
print(len(stations_df))

display(
    stations_df.head(40)
)

Colunas:
['estacao_id', 'estacao', 'latitude', 'longitude', 'cota_m', 'utm_x_sad69z23', 'utm_y_sad69z23', 'tem_meteorologica']

Quantidade de estações cadastradas:
33


,estacao_id,estacao,latitude,longitude,cota_m,utm_x_sad69z23,utm_y_sad69z23,tem_meteorologica
0,1,Vidigal,-22.99250,-43.23306,85,681138.532,7456241.298,True
1,2,Urca,-22.95583,-43.16667,90,688004.213,7460236.157,False
2,3,Rocinha,-22.98583,-43.24500,160,679831.802,7457041.035,False
3,4,Tijuca,-22.93194,-43.22167,340,682358.108,7462941.416,False
4,5,Santa Teresa,-22.93167,-43.19639,170,684951.792,7462971.838,False
5,6,Copacabana,-22.98639,-43.18944,90,685675.030,7456902.449,False
6,7,Grajaú,-22.92222,-43.26750,80,677639.269,7463809.403,False
7,8,Ilha do Governador,-22.81806,-43.21028,0,683708.659,7475959.609,False
8,9,Penha,-22.84444,-43.27528,111,677059.917,7472757.104,False
9,10,Madureira,-22.87333,-43.33889,45,670409.679,7469665.020,False


In [4]:
colunas_necessarias = [
    "estacao_id",
    "estacao",
    "latitude",
    "longitude",
]

colunas_ausentes = [
    coluna
    for coluna in colunas_necessarias
    if coluna not in stations_df.columns
]

if colunas_ausentes:
    raise ValueError(
        f"Colunas ausentes no cadastro: {colunas_ausentes}"
    )

stations_df = stations_df[
    colunas_necessarias
].copy()

stations_df["estacao_id"] = pd.to_numeric(
    stations_df["estacao_id"],
    errors="coerce",
)

stations_df["latitude"] = pd.to_numeric(
    stations_df["latitude"],
    errors="coerce",
)

stations_df["longitude"] = pd.to_numeric(
    stations_df["longitude"],
    errors="coerce",
)

estacoes_invalidas = stations_df[
    stations_df[
        [
            "estacao_id",
            "latitude",
            "longitude",
        ]
    ].isna().any(axis=1)
]

if not estacoes_invalidas.empty:
    print("Estações com identificador ou coordenadas inválidas:")
    display(estacoes_invalidas)

stations_df = (
    stations_df
    .dropna(
        subset=[
            "estacao_id",
            "latitude",
            "longitude",
        ]
    )
    .drop_duplicates(
        subset=["estacao_id"]
    )
    .copy()
)

stations_df["estacao_id"] = (
    stations_df["estacao_id"]
    .astype(int)
)

print(
    "Estações com coordenadas válidas:",
    len(stations_df),
)

display(stations_df)

Estações com coordenadas válidas: 33


,estacao_id,estacao,latitude,longitude
0,1,Vidigal,-22.99250,-43.23306
1,2,Urca,-22.95583,-43.16667
2,3,Rocinha,-22.98583,-43.24500
3,4,Tijuca,-22.93194,-43.22167
4,5,Santa Teresa,-22.93167,-43.19639
5,6,Copacabana,-22.98639,-43.18944
6,7,Grajaú,-22.92222,-43.26750
7,8,Ilha do Governador,-22.81806,-43.21028
8,9,Penha,-22.84444,-43.27528
9,10,Madureira,-22.87333,-43.33889


In [6]:
arquivos = sorted(
    pasta_alertario.glob("*.parquet")
)

print(
    "Quantidade de arquivos:",
    len(arquivos),
)

estacoes_encontradas = []

for i, arquivo in enumerate(
    arquivos,
    start=1,
):

    df = pd.read_parquet(
        arquivo,
        columns=[
            "estacao_id",
            "estacao",
        ],
    )

    estacoes_encontradas.append(
        df[
            [
                "estacao_id",
                "estacao",
            ]
        ].drop_duplicates()
    )

    if i % 50 == 0 or i == len(arquivos):
        print(
            f"Arquivos analisados: {i}/{len(arquivos)}"
        )

estacoes_parquet = (
    pd.concat(
        estacoes_encontradas,
        ignore_index=True,
    )
    .drop_duplicates()
    .sort_values("estacao_id")
    .reset_index(drop=True)
)

print(
    "\nEstações presentes nos arquivos pluviométricos:",
    len(estacoes_parquet),
)

display(estacoes_parquet)

Quantidade de arquivos: 471
Arquivos analisados: 50/471
Arquivos analisados: 100/471
Arquivos analisados: 150/471
Arquivos analisados: 200/471
Arquivos analisados: 250/471
Arquivos analisados: 300/471
Arquivos analisados: 350/471
Arquivos analisados: 400/471
Arquivos analisados: 450/471
Arquivos analisados: 471/471

Estações presentes nos arquivos pluviométricos: 39


,estacao_id,estacao
0,1,Vidigal
1,2,Urca
2,3,Rocinha
3,4,Tijuca
4,5,Santa Teresa
5,6,Copacabana
6,7,Grajaú
7,8,Ilha do Governador
8,9,Penha
9,10,Madureira


In [7]:
ids_cadastro = set(
    stations_df["estacao_id"]
)

ids_parquet = set(
    estacoes_parquet["estacao_id"]
)

print(
    "Estações no cadastro:",
    len(ids_cadastro),
)

print(
    "Estações nos arquivos Parquet:",
    len(ids_parquet),
)

print(
    "Estações presentes nos dois:",
    len(
        ids_cadastro & ids_parquet
    ),
)

print(
    "IDs no cadastro, mas sem dados pluviométricos:",
    sorted(
        ids_cadastro - ids_parquet
    ),
)

print(
    "IDs com dados pluviométricos, mas sem cadastro:",
    sorted(
        ids_parquet - ids_cadastro
    ),
)

stations_df = stations_df[
    stations_df["estacao_id"].isin(
        ids_parquet
    )
].copy()

print(
    "\nEstações que serão mapeadas:",
    len(stations_df),
)

display(stations_df)

Estações no cadastro: 33
Estações nos arquivos Parquet: 39
Estações presentes nos dois: 33
IDs no cadastro, mas sem dados pluviométricos: []
IDs com dados pluviométricos, mas sem cadastro: [34, 100, 101, 102, 103, 104]

Estações que serão mapeadas: 33


,estacao_id,estacao,latitude,longitude
0,1,Vidigal,-22.99250,-43.23306
1,2,Urca,-22.95583,-43.16667
2,3,Rocinha,-22.98583,-43.24500
3,4,Tijuca,-22.93194,-43.22167
4,5,Santa Teresa,-22.93167,-43.19639
5,6,Copacabana,-22.98639,-43.18944
6,7,Grajaú,-22.92222,-43.26750
7,8,Ilha do Governador,-22.81806,-43.21028
8,9,Penha,-22.84444,-43.27528
9,10,Madureira,-22.87333,-43.33889


In [8]:
grid = np.load(
    arquivo_grade_radar
)

lat_grid = grid["lat"]
lon_grid = grid["lon"]

print(
    "Shape da grade de latitude:",
    lat_grid.shape,
)

print(
    "Shape da grade de longitude:",
    lon_grid.shape,
)

if lat_grid.shape != lon_grid.shape:
    raise ValueError(
        "As grades de latitude e longitude "
        "possuem dimensões diferentes."
    )

Shape da grade de latitude: (654, 656)
Shape da grade de longitude: (654, 656)


In [10]:
mapping = []

for _, row in stations_df.iterrows():

    station_id = row["estacao_id"]

    station_name = row["estacao"]

    station_lat = row["latitude"]

    station_lon = row["longitude"]

    # Distância da estação para cada pixel da grade.
    dist = (
        (lat_grid - station_lat) ** 2
        + (lon_grid - station_lon) ** 2
    )

    # Posição do pixel mais próximo.
    pixel_i, pixel_j = np.unravel_index(
        np.nanargmin(dist),
        dist.shape,
    )

    mapping.append(
        {
            "station_id": int(station_id),
            "nome": station_name,
            "latitude": float(station_lat),
            "longitude": float(station_lon),
            "pixel_i": int(pixel_i),
            "pixel_j": int(pixel_j),
        }
    )

mapping_df = (
    pd.DataFrame(mapping)
    .sort_values("station_id")
    .reset_index(drop=True)
)

print(
    "Quantidade de estações mapeadas:",
    len(mapping_df),
)

Quantidade de estações mapeadas: 33


In [11]:
mapping_df.to_csv(
    arquivo_saida,
    index=False,
)

print(
    "Arquivo salvo em:",
    arquivo_saida.resolve(),
)

print(
    "Quantidade de estações:",
    len(mapping_df),
)

display(
    mapping_df.head(40)
)

Arquivo salvo em: /home/noemi/atmoseer/notebooks/sumare_radar/mapeamento_pixel_estacao_alertario.csv
Quantidade de estações: 33


,station_id,nome,latitude,longitude,pixel_i,pixel_j
0,1,Vidigal,-22.99250,-43.23306,336,331
1,2,Urca,-22.95583,-43.16667,327,347
2,3,Rocinha,-22.98583,-43.24500,334,328
3,4,Tijuca,-22.93194,-43.22167,320,334
4,5,Santa Teresa,-22.93167,-43.19639,320,340
5,6,Copacabana,-22.98639,-43.18944,335,342
6,7,Grajaú,-22.92222,-43.26750,318,323
7,8,Ilha do Governador,-22.81806,-43.21028,291,337
8,9,Penha,-22.84444,-43.27528,298,321
9,10,Madureira,-22.87333,-43.33889,305,306
